# Duel fin : `ml_15` contre `ml_14`, validation croisée répétée

**Pourquoi** : le duel prévu à l'avance (`ml_15`, colonnes de `ml_11`, contre `ml_14`, base `ml_12`, mêmes grilles élargies) est **dans le bruit** en meilleur contre meilleur sur les 5 plis des scénarios (CatBoost de `ml_14` → régression logistique de `ml_15` : gain de 0,004, 3 plis sur 5). Avec 5 plis, la comparaison n'a pas la puissance de trancher un écart de cette taille : « dans le bruit » veut dire « pas démontré », pas « moins bon ». Plutôt que de changer la règle après coup, ce notebook **mesure plus finement**, avec l'outil préparé pour l'étape 5 (`ml_16`).

**Ce n'est pas un scénario** : rien n'est écrit dans le fichier de résultats, aucun modèle n'est enregistré, le test n'est pas lu.

**Règle, fixée avant de lancer (07/10)** :
1. **Opposés** : le premier du podium de chaque scénario, relu dans le fichier de résultats (rang 1), chacun avec **ses variables** et **ses réglages figés** (ceux retenus par la boucle) : pas de nouvelle recherche d'hyperparamètres.
2. **Validation croisée répétée** : 5 plis × 10 répétitions (`RepeatedStratifiedKFold`, `random_state=42`), soit 50 comparaisons pli par pli ; les deux versions voient exactement les mêmes plis.
3. **Mesure** : le score décisionnel F2 au rappel minimal de 60 % (celui des scénarios) ; la précision moyenne en contrôle.
4. **Verdict** : **test t corrigé de Nadeau et Bengio** sur les 50 gains (`ml_15` moins `ml_14`) : t = gain moyen / racine((1 / 50 + 1 / 4) × variance des gains). La correction tient compte de ce que les répétitions réutilisent les mêmes clients.
5. **Décision** : **t ≥ 2** en meilleur contre meilleur → la version de `ml_15` devient la version finale ; sinon `ml_14` reste, et le sujet est clos. Les comparaisons modèle par modèle sont données pour information, elles ne changent pas la décision.

**Données** : `data/ML/train_dataset.csv` (clients au contentieux à M retirés, 80 % des clients restants). Le test n'est pas lu.

In [1]:
# ==============================================================================
# RÉGLAGES DU LANCEMENT
# ==============================================================================
SCENARIO_A, SCENARIO_B = "14", "15"   # référence (A) et challenger (B) : gain = B - A
N_PLIS, N_REPETITIONS = 5, 10
SEUIL_T = 2.0
VERSION_METHODE = "v8"
RAPPEL_MINIMAL = 0.60

## 1. Données, variables et réglages figés des deux scénarios

In [2]:
import ast
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", None)

current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())
train = pd.read_csv(root_dir / "data" / "ML" / "train_dataset.csv")
y = train["dpnm"].to_numpy()

# Variables de chaque scénario, rangées par encodage (copie des cellules « variables du scénario » de ml_14 et ml_15)
socle = {"num": ["LIMIT_BAL", "AGE"] + [f"PAY_{n}" for n in range(1, 7)], "log": [f"PAY_AMT{n}" for n in range(1, 7)],
         "log_signe": [f"BILL_AMT{n}" for n in range(1, 7)], "nom": ["SEX", "EDUCATION", "MARRIAGE"]}
SCENARIOS = {
    "14": {"retirees": ["AGE", "SEX", "EDUCATION", "MARRIAGE"],
           "ajout": {"num": ["CUMUL_INCIDENT"], "log": [], "log_signe": [], "nom": ["PAY_habituel"]}},
    "15": {"retirees": [],
           "ajout": {"num": ["ratio_PAY_BILL_global", "ratio_PAY_BILL_median", "ratio_PAY_BILL_regularite", "ratio_PAY_BILL1", "ratio_PAY_BILL2",
                             "ratio_PAY_BILL3", "ratio_PAY_BILL4", "ratio_PAY_BILL5", "ratio_BILL_LIMIT1", "ratio_BILL_LIMIT2", "ratio_BILL_LIMIT3",
                             "ratio_BILL_LIMIT4", "ratio_BILL_LIMIT5", "ratio_BILL_LIMIT6", "NB_MOIS_CTX", "CUMUL_INCIDENT"],
                     "log": ["BILL_evol_relatif", "PAYAMT_evol_relatif"], "log_signe": ["BILL_evol_brut", "PAYAMT_evol_brut"],
                     "nom": ["FLAG_OUVERTURE", "FLAG_DEGEL", "MOIS_ACTIVATION", "TYPE_USAGE", "PAY_habituel", "FAUX_CODAGE", "SURVEILLANCE_RECENTE",
                             "FLAG_CTX", "MOIS_SORTIE_CTX", "FLAG_RETARD", "MOIS_SORTIE_RETARD"]}},
}
remplacements = {
    "ratio_PAY_BILL_global": ([np.nan], 100),
    "ratio_PAY_BILL_median": ([-1], 100),
    "ratio_PAY_BILL_regularite": ([np.nan, -1], 100),
}
TYPES = {s: {t: [c for c in socle[t] if c not in d["retirees"]] + d["ajout"][t] for t in socle} for s, d in SCENARIOS.items()}
for s, types in TYPES.items():
    absentes = [c for c in sum(types.values(), []) if c not in train.columns]
    assert not absentes, f"Colonnes absentes du train : {absentes}"

def lire_reglage(texte):
    # « depth 4 ; iterations 297 ; … » -> {"depth": 4, "iterations": 297, …}
    reglage = {}
    for morceau in texte.split(" ; "):
        nom, valeur = morceau.split(" ", 1)
        try:
            reglage[nom] = ast.literal_eval(valeur)
        except (ValueError, SyntaxError):
            reglage[nom] = valeur
    return reglage

resultats = pd.read_csv(root_dir / "data" / "ML" / "resultats_scenarios.csv")
REGLAGES, PREMIER = {}, {}
for s in (SCENARIO_A, SCENARIO_B):
    lignes = resultats[(resultats["scenario"] == f"ml_{s}") & (resultats["version_methode"] == VERSION_METHODE)]
    assert not lignes.empty, f"ml_{s} n'a pas de résultats enregistrés avec la méthode {VERSION_METHODE}"
    assert (lignes["rappel_minimal"] == RAPPEL_MINIMAL).all()
    REGLAGES[s] = {l["modele"]: lire_reglage(l["reglage"]) for _, l in lignes.iterrows()}
    PREMIER[s] = lignes.loc[lignes["rang"] == 1, "modele"].iloc[0]
MODELES = [m for m in REGLAGES[SCENARIO_A] if m in REGLAGES[SCENARIO_B]]

display(Markdown(f"**Opposés (premier du podium de chaque scénario)** : `ml_{SCENARIO_A}` {PREMIER[SCENARIO_A]} "
                 f"({len(sum(TYPES[SCENARIO_A].values(), []))} variables) contre `ml_{SCENARIO_B}` {PREMIER[SCENARIO_B]} "
                 f"({len(sum(TYPES[SCENARIO_B].values(), []))} variables)"))
display(pd.DataFrame({"Modèle": MODELES,
                      f"Réglage figé, ml_{SCENARIO_A}": [resultats[(resultats["scenario"] == f"ml_{SCENARIO_A}") & (resultats["version_methode"] == VERSION_METHODE) & (resultats["modele"] == m)]["reglage"].iloc[0] for m in MODELES],
                      f"Réglage figé, ml_{SCENARIO_B}": [resultats[(resultats["scenario"] == f"ml_{SCENARIO_B}") & (resultats["version_methode"] == VERSION_METHODE) & (resultats["modele"] == m)]["reglage"].iloc[0] for m in MODELES]}).set_index("Modèle"))

**Opposés (premier du podium de chaque scénario)** : `ml_14` CatBoost (21 variables) contre `ml_15` LogisticRegression (54 variables)

,"Réglage figé, ml_14","Réglage figé, ml_15"
Modèle,,
CatBoost,depth 4 ; iterations 297 ; l2_leaf_reg 6.67 ; learning_rate 0.00887,depth 2 ; iterations 445 ; l2_leaf_reg 10.0 ; learning_rate 0.00887
RandomForest,max_depth 6 ; max_features sqrt ; min_samples_leaf 225 ; n_estimators 300,max_depth 4 ; max_features sqrt ; min_samples_leaf 225 ; n_estimators 300
LogisticRegression,C 0.1 ; l1_ratio 0.0,C 0.1 ; l1_ratio 1.0
MLPClassifier,"alpha 1.0 ; hidden_layer_sizes (128, 64)","alpha 1.0 ; hidden_layer_sizes (128, 64)"


In [3]:
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

scale_pos = (y == 0).sum() / (y == 1).sum()
TRANSFORMATIONS = {
    "num": lambda: StandardScaler(),
    "log": lambda: make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()),
    "log_signe": lambda: make_pipeline(FunctionTransformer(np.arcsinh, feature_names_out="one-to-one"), StandardScaler()),
    "nom": lambda: OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
}

def avec_remplacement(colonne, transformation):
    valeurs, valeur = remplacements[colonne]
    valeurs = sorted(valeurs, key=lambda v: not pd.isna(v))
    return make_pipeline(*[SimpleImputer(missing_values=v, strategy="constant", fill_value=valeur) for v in valeurs], transformation)

def preprocesseur(types):
    # Le pipeline des scénarios (méthode v8), limité aux colonnes du scénario
    return ColumnTransformer(transformers=
        [(t, TRANSFORMATIONS[t](), [c for c in cols if c not in remplacements]) for t, cols in types.items() if [c for c in cols if c not in remplacements]]
        + [(f"remplacement_{c}", avec_remplacement(c, TRANSFORMATIONS[t]()), [c]) for t, cols in types.items() for c in cols if c in remplacements])

def modele(nom, scenario):
    # Modèle aux réglages figés du scénario ; un seul thread par modèle, les plis tournant en parallèle
    r = REGLAGES[scenario][nom]
    estimateurs = {
        "LogisticRegression": lambda: LogisticRegression(solver="saga", max_iter=5000, class_weight="balanced", random_state=42, **r),
        "MLPClassifier": lambda: MLPClassifier(max_iter=500, random_state=42, early_stopping=True, **r),
        "RandomForest": lambda: RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=1, **r),
        "CatBoost": lambda: CatBoostClassifier(scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=1, allow_writing_files=False, **r),
    }
    return make_pipeline(preprocesseur(TYPES[scenario]), estimateurs[nom]())

## 2. Validation croisée répétée : les deux scénarios, sur les mêmes plis

Chaque modèle de chaque scénario est entraîné sur chacun des 50 découpages et jugé sur le pli de validation (score décisionnel F2 au rappel minimal de 60 %, précision moyenne). Une ligne s'affiche par modèle, avec sa durée.

In [4]:
import time
from joblib import Parallel, delayed
from sklearn.metrics import average_precision_score, fbeta_score
from sklearn.model_selection import RepeatedStratifiedKFold

def seuil_pour_rappel(y_vrai, proba, rappel=RAPPEL_MINIMAL):
    proba_defauts = np.sort(proba[y_vrai == 1])[::-1]
    return proba_defauts[int(np.ceil(rappel * len(proba_defauts))) - 1]

def f2_au_rappel_minimal(y_vrai, proba):
    return fbeta_score(y_vrai, proba >= seuil_pour_rappel(y_vrai, proba), beta=2)

plis = list(RepeatedStratifiedKFold(n_splits=N_PLIS, n_repeats=N_REPETITIONS, random_state=42).split(train, y))

def un_pli(nom, scenario, entrainement, validation):
    colonnes = sum(TYPES[scenario].values(), [])
    m = modele(nom, scenario).fit(train.iloc[entrainement][colonnes], y[entrainement])
    p = m.predict_proba(train.iloc[validation][colonnes])[:, 1]
    return f2_au_rappel_minimal(y[validation], p), average_precision_score(y[validation], p)

scores = {}   # (scénario, modèle) -> {"F2": [...], "Précision moyenne": [...]}
for nom in MODELES:
    debut = time.time()
    for s in (SCENARIO_A, SCENARIO_B):
        res = Parallel(n_jobs=-1)(delayed(un_pli)(nom, s, e, v) for e, v in plis)
        scores[(s, nom)] = {"F2": np.array([r[0] for r in res]), "Précision moyenne": np.array([r[1] for r in res])}
    print(f"{nom} : F2 ml_{SCENARIO_A} {scores[(SCENARIO_A, nom)]['F2'].mean():.4f}, ml_{SCENARIO_B} {scores[(SCENARIO_B, nom)]['F2'].mean():.4f} "
          f"({(time.time() - debut) / 60:.1f} min)")

CatBoost : F2 ml_14 0.4844, ml_15 0.4855 (0.3 min)
RandomForest : F2 ml_14 0.4833, ml_15 0.4835 (0.4 min)
LogisticRegression : F2 ml_14 0.4817, ml_15 0.4870 (0.1 min)
MLPClassifier : F2 ml_14 0.4800, ml_15 0.4845 (0.2 min)


## 3. Verdict : test t corrigé de Nadeau et Bengio

Pour le meilleur contre meilleur (la décision), puis pour chaque modèle (pour information) : gain moyen (`ml_15` moins `ml_14`), écart-type sur les 50 plis, plis gagnés, t corrigé et verdict. **Gain réel** si t ≥ 2, **perte** si t ≤ -2, sinon **dans le bruit**.

In [5]:
CORRECTION = 1 / len(plis) + 1 / (N_PLIS - 1)   # Nadeau et Bengio : 1/J + taille de validation / taille d'entraînement

def t_corrige(gains):
    variance = gains.var(ddof=1)
    return gains.mean() / np.sqrt(CORRECTION * variance) if variance > 0 else 0.0

def verdict(t):
    return "✅ gain réel" if t >= SEUIL_T else ("❌ perte" if t <= -SEUIL_T else "➖ dans le bruit")

comparaisons = [(f"Meilleur contre meilleur ({PREMIER[SCENARIO_A]} ml_{SCENARIO_A} → {PREMIER[SCENARIO_B]} ml_{SCENARIO_B})",
                 PREMIER[SCENARIO_A], PREMIER[SCENARIO_B])] + [(nom, nom, nom) for nom in MODELES]
lignes = []
for libelle, nom_a, nom_b in comparaisons:
    ligne = {"Comparaison": libelle}
    for mesure in ("F2", "Précision moyenne"):
        a, b = scores[(SCENARIO_A, nom_a)][mesure], scores[(SCENARIO_B, nom_b)][mesure]
        gains = b - a
        t = t_corrige(gains)
        ligne[f"{mesure} ml_{SCENARIO_A}"] = round(a.mean(), 4)
        ligne[f"{mesure} ml_{SCENARIO_B}"] = round(b.mean(), 4)
        ligne[f"{mesure} : gain"] = f"{gains.mean():+.4f} ± {gains.std(ddof=1):.4f}"
        ligne[f"{mesure} : plis gagnés"] = f"{(gains > 0).sum()} sur {len(gains)}"
        ligne[f"{mesure} : t corrigé"] = round(t, 2)
        ligne[f"{mesure} : verdict"] = verdict(t)
    lignes.append(ligne)
tableau = pd.DataFrame(lignes).set_index("Comparaison")
display(Markdown(f"**`ml_{SCENARIO_B}` contre `ml_{SCENARIO_A}`, pli par pli ({len(plis)} plis, réglages figés)**"))
display(tableau[[c for c in tableau.columns if c.startswith("F2")]])
display(tableau[[c for c in tableau.columns if c.startswith("Précision moyenne")]])
decision = tableau.iloc[0]["F2 : verdict"]
display(Markdown(f"**Décision (meilleur contre meilleur, score décisionnel F2)** : {decision} → "
                 + (f"**la version de `ml_{SCENARIO_B}` ({PREMIER[SCENARIO_B]}) devient la version finale**" if decision == "✅ gain réel"
                    else f"**`ml_{SCENARIO_A}` reste la version de base, sujet clos**")))

**`ml_15` contre `ml_14`, pli par pli (50 plis, réglages figés)**

,F2 ml_14,F2 ml_15,F2 : gain,F2 : plis gagnés,F2 : t corrigé,F2 : verdict
Comparaison,,,,,,
Meilleur contre meilleur (CatBoost ml_14 → LogisticRegression ml_15),0.4844,0.4870,+0.0026 ± 0.0063,31 sur 50,0.79,➖ dans le bruit
CatBoost,0.4844,0.4855,+0.0010 ± 0.0042,28 sur 50,0.47,➖ dans le bruit
RandomForest,0.4833,0.4835,+0.0003 ± 0.0029,26 sur 50,0.20,➖ dans le bruit
LogisticRegression,0.4817,0.4870,+0.0054 ± 0.0062,40 sur 50,1.65,➖ dans le bruit
MLPClassifier,0.4800,0.4845,+0.0045 ± 0.0072,36 sur 50,1.19,➖ dans le bruit


,Précision moyenne ml_14,Précision moyenne ml_15,Précision moyenne : gain,Précision moyenne : plis gagnés,Précision moyenne : t corrigé,Précision moyenne : verdict
Comparaison,,,,,,
Meilleur contre meilleur (CatBoost ml_14 → LogisticRegression ml_15),0.3359,0.3336,-0.0023 ± 0.0073,16 sur 50,-0.62,➖ dans le bruit
CatBoost,0.3359,0.3337,-0.0022 ± 0.0045,11 sur 50,-0.96,➖ dans le bruit
RandomForest,0.3337,0.3332,-0.0005 ± 0.0036,22 sur 50,-0.29,➖ dans le bruit
LogisticRegression,0.3327,0.3336,+0.0008 ± 0.0061,31 sur 50,0.26,➖ dans le bruit
MLPClassifier,0.3304,0.3359,+0.0056 ± 0.0078,37 sur 50,1.36,➖ dans le bruit


**Décision (meilleur contre meilleur, score décisionnel F2)** : ➖ dans le bruit → **`ml_14` reste la version de base, sujet clos**

**Comment lire** : un gain positif veut dire que `ml_15` fait mieux que `ml_14`. Avec 50 plis, le t corrigé dit si le gain moyen dépasse ce que le hasard des découpages peut produire. La décision s'écrit ensuite dans le journal `tableau_ML.md` (lignes de `ml_14` et `ml_15`) et dans la méthodologie. Si `ml_15` l'emporte, la base de l'étape 5 (`ml_16`, `BASE`) passe à `"15"`.